# PROTOTYPE EQUITY PRICING

One of the basic functions of OSEM is creating and manipulating cash flows from different classes of financial instruments. This page shows the processing of company shares held within a portfolio.

An ownership interest or a share represents ownership of a small piece of a legal entity. The most common of them being listed companies. In OSEM these shares provide two kinds of benefits. A periotic dividend and its intrinsic value (The share can be sold to another entity).

The enterprise and with it the equity share is assumed to grow at a constant rate every year. At periodic periods, the dividend is paid out. The size of the dividend determined by the value of the share. A % of the market value referred here as dividend yield.

This page has 4 sections. 

 1) The first section shows how to import the necessary information about equity positions, the overall economic environment as well as other parameters. 
2) The next section imports the term structure which is needed later on. 
3) The third section shows how to generate the cash flows for a set of equities. 
4) The last section shows how a hypothetical equity can be calibrated if the user wishes to preform a risk-neutral run.



## Importing and handling of equity data

The packages and imports needed in this script are the following:

In [75]:
import datetime
import os
import pandas as pd

from ConfigurationClass import Configuration
from CurvesClass import Curves
from EquityClasses import *
from ImportData import get_EquityShare, get_settings, import_SWEiopa, \
    get_configuration
from MainLoop import create_cashflow_dataframe

Set up the base folder

In [76]:
base_folder = os.getcwd()  # Get current working directory

Most of the run settings are saved in the configuration file:

In [77]:
conf: Configuration
conf = get_configuration(os.path.join(base_folder, "ALM.ini"), os)

These lines of code just extract the absolute location of different files:

In [78]:
parameters_file = conf.input_parameters
cash_portfolio_file = conf.input_cash_portfolio
equity_portfolio_file = conf.input_equity_portfolio

The settings object holds data about file locations, information about the run settings and model parameters such as modelling date.

In [79]:
settings = get_settings(parameters_file)

The EquityShare object contains information about each equity position. This includes:
 * Asset_ID
 * NACE
 * Issuer
 * Issue_Date
 * Dividend_Yield
 * Frequency
 * Units
 * Market_Price
 * Growth_Rate
  

In [80]:
equity_input_generator = get_EquityShare(equity_portfolio_file)
equity_input = {equity_share.asset_id: equity_share for equity_share in equity_input_generator}

EquitySharePortfolio object contains all EquityShare objects in a dictionary.

In [81]:
equity_portfolio = EquitySharePortfolio(equity_input)

## Importing the information about the economic environment

import_SWEiopa() reads the necessary data about the current yield curve. One of these parameters (the ufr or ultimate forward rate) is necessary in the equity example as ufr is used in the Gordon growth formula to calculate the terminal value of the equity position. Inside OSEM, the parameters related to the yield curve are saved in the Curves object. 

In [82]:
[maturities_country, curve_country, extra_param, Qb] = import_SWEiopa(settings.EIOPA_param_file,
                                                                          settings.EIOPA_curves_file, settings.country)
# Curves object with information about term structure
curves = Curves(extra_param["UFR"] / 100, settings.precision, settings.tau, settings.modelling_date,
                settings.country)

In [83]:
ufr = extra_param["UFR"]/100 # ultimate forward rate
precision = float(settings.precision) # Numeric precision of the optimisation
# Targeted distance between the extrapolated curve and the ufr at the convergence point
tau = float(settings.tau) # 1 basis point

In [84]:
curves.SetObservedTermStructure(maturity_vec=curve_country.index.tolist(), yield_vec=curve_country.values)
curves.CalcFwdRates()

In [85]:
curves.ProjectForwardRate(settings.n_proj_years)

In [86]:
curves.CalibrateProjected(settings.n_proj_years, 0.05, 0.5, 1000)

## Projection of cash flows for an equity portfolio

The first computation step inside the OSEM equity preparation process is the identification of all the unique dates and dividend size amounts. The representation of assets in terms of individual cash flows on the time-line is one of the core principles of OSEM. This is done by two functions. One for dividend dates and another for terminal rates.

Both functions generate a list of dictionaries containing the date of a cash flow and the amount. Same is also true for the terminal amount calculation. 

#### Calculation of the dividend amount:

The dividend size is calculated using the dividend yield provided as input for each equity position. However the market value changes as time moves forward. To account for this, the growth rate and the time fraction between the modelling date and the date of the cash flow is used to calculate a future market value.

ToDo Formulas

The same logic is applied to the calculation of terminal rates.

In [87]:
dividend_flows = equity_portfolio.create_dividend_flows(settings.modelling_date, settings.end_date)
terminal_flows = equity_portfolio.create_terminal_flows(modelling_date=settings.modelling_date,
                                                            terminal_date=settings.end_date,
                                                            terminal_rate=curves.ufr)

All cash flows can be represented in a matrix with all possible cash flow dates as columns and all equities as rows. The non-zero entries then represent the value of the cash flow at that date. The first step is to calculate the unique dates for the entire portfolio of equities. This is done by the unique_dates_profiles() function.

The same logic can be applied to terminal dates. 

Both can then conveniently be represented as DataFrames.

Note that a vector of growth rates is also provided as output. This makes it simpler to increase the market value of the portfolio as OSEM moves from one modelling period to the next one.

In [88]:
unique_list = equity_portfolio.unique_dates_profile(dividend_flows)
unique_terminal_list = equity_portfolio.unique_dates_profile(terminal_flows)

In [89]:
[market_price_df, growth_rate_df, units_df] = equity_portfolio.init_equity_portfolio_to_dataframe(settings.modelling_date)

The create_cashflow_dataframe() function converts the list of dictionaries of cashflows and dates, into a single DataFrame:

In [90]:
cash_flows = create_cashflow_dataframe(dividend_flows, unique_list)
# Dataframe with terminal cash flows
terminal_cash_flows = create_cashflow_dataframe(terminal_flows, unique_terminal_list)

In [91]:
display(cash_flows)

,2023-12-03,2024-12-03,2025-12-03,2026-12-03,2027-12-03,2028-12-03,2029-12-03,2030-12-03,2031-12-03,2032-12-03,...,2063-12-03,2064-12-03,2065-12-03,2066-12-03,2067-12-03,2068-12-03,2069-12-03,2070-12-03,2071-12-03,2072-12-03
1125,3.712281,3.749481,3.786950,3.824793,3.863015,3.901725,3.940715,3.980095,4.019869,4.060150,...,5.527081,5.582466,5.638252,5.694596,5.751503,5.809137,5.867188,5.925819,5.985037,6.045011
2123,0.613205,0.625494,0.637996,0.650747,0.663753,0.677055,0.690587,0.704389,0.718467,0.732866,...,1.353981,1.381117,1.408720,1.436875,1.465592,1.494965,1.524844,1.555319,1.586404,1.618198
3232,2.315166,2.407967,2.504218,2.604317,2.708417,2.816981,2.929581,3.046683,3.168465,3.295469,...,11.115161,11.560698,12.022803,12.503380,13.003166,13.524382,14.064979,14.627186,15.211865,15.821613
1,2.323760,2.347045,2.370499,2.394188,2.418114,2.442345,2.466751,2.491402,2.516299,2.541514,...,3.459761,3.494430,3.529351,3.564620,3.600242,3.636318,3.672656,3.709358,3.746426,3.783967
2,3.946368,4.025460,4.105913,4.187975,4.271676,4.357287,4.444372,4.533198,4.623800,4.716467,...,8.713738,8.888374,9.066019,9.247214,9.432030,9.621062,9.813350,10.009482,10.209533,10.414147
3,0.050365,0.052384,0.054478,0.056656,0.058920,0.061282,0.063732,0.066279,0.068928,0.071691,...,0.241805,0.251497,0.261550,0.272005,0.282878,0.294216,0.305977,0.318207,0.330927,0.344191
4,0.094158,0.095101,0.096051,0.097011,0.097981,0.098963,0.099951,0.100950,0.101959,0.102981,...,0.140188,0.141593,0.143007,0.144437,0.145880,0.147342,0.148814,0.150301,0.151803,0.153324
5,2.282822,2.328574,2.375113,2.422582,2.471000,2.520523,2.570898,2.622281,2.674690,2.728295,...,5.040562,5.141583,5.244343,5.349158,5.456067,5.565414,5.676646,5.790100,5.905822,6.024184
6,3.930951,4.088518,4.251945,4.421904,4.598656,4.782988,4.974174,5.173002,5.379777,5.595419,...,18.872575,19.629059,20.413674,21.229651,22.078244,22.963223,23.881111,24.835688,25.828423,26.863723
7,2.836797,2.865224,2.893856,2.922775,2.951983,2.981564,3.011359,3.041452,3.071845,3.102627,...,4.223605,4.265928,4.308558,4.351614,4.395100,4.439142,4.483503,4.528307,4.573559,4.619389


In [92]:
display(terminal_cash_flows)

,2073-04-29
1125,202.308916
2123,32.622931
3232,401.833450
1,126.638375
2,209.949558
3,8.741690
4,5.131321
5,121.447744
6,682.278232
7,154.597497


#### Risk spreads 

In OSEM, the cash flows of equity shares increase with the assumed growth rate. However in real world applications, when calculating the present value of these cash flows, the "riskiness" of the issuer should be accounted for. This is done using discounting spreads. These spreads represent the fact that certain geographies and certain sectors are more prone to failure and the required risk premium is therefore higher.

Each equity position has 3 kinds of spreads are available:

 1) Country specific spread: Country specific spread represents the extra riskiness related to the country of operation (spread_country)
 2) Sector specific spread: Spread specific to the sector in which the issuer opperates (spread_sector)
 3) Assumed shock scenario specific spread: Spread specific to the shock scenario (spread_stress)

#### Calculation of present value of each instrument

The cashflows can be used to price the current market value of the bond, implied by the assumed economic parameters.

Note that this pricing is done using the risk free rate as the discounting factor with a risk spread equal to the sum of the 3 components. This combined spread is added to the yield in a parralel shift. To demonstrate, the discount factors are calculated as:

$$
df_i (t) = \frac{1}{(y(t) + spreadCountry + spreadSector+spreadShock)^t}
$$

Where:
 - $t$ is the time period
 - $df_i(t)$ is the discount factor at time $t$ for the equity $i$
 - $y(t)$ is the yield implied by the risk-free interest rate at time $t$
 - $spreadCountry$, $spreadSector$, $spreadShock$ are the country, sector and shock spreads.


This example will show pricing at the modelling date.

In [93]:
proj_period = 0

In [94]:
asset_id_tmp = cash_flows.index[0]

In [95]:
equity_portfolio.equity_share[asset_id_tmp]

EquityShare(asset_id=1125, nace='A1.4.5', issuer=None, issue_date=datetime.date(2021, 12, 3), dividend_yield=0.03, frequency=1, units=100.0, market_price=123.01, growth_rate=0.01, spread_country=0.0, spread_sector=0.0, spread_stress=0.0)

In [96]:
asset_id_tmp = cash_flows.index[0]
    
temp_dividend = cash_flows.loc[asset_id_tmp].to_dict() 

temp_terminal = terminal_cash_flows.loc[asset_id_tmp].to_dict()

price = equity_portfolio.equity_share[asset_id_tmp].price_share(temp_dividend, temp_terminal, settings.modelling_date, proj_period, curves)
print(price)

[144.37329342]


#### Additivity of spreads

The 3 kinds of spread work in the same way and increasing any of them has the same effect.

We will demonstrate this by fixing them one at a time.


In [97]:
asset_id_tmp = cash_flows.index[0]
    
temp_dividend = cash_flows.loc[asset_id_tmp].to_dict() 

temp_terminal = terminal_cash_flows.loc[asset_id_tmp].to_dict()

In [98]:
equity_portfolio.equity_share[asset_id_tmp].spread_country = 0.01
equity_portfolio.equity_share[asset_id_tmp].spread_sector = 0.01
equity_portfolio.equity_share[asset_id_tmp].spread_stress = 0.00

In [99]:
price = equity_portfolio.equity_share[asset_id_tmp].price_share(temp_dividend, temp_terminal, settings.modelling_date, proj_period, curves)
print(price)

[89.59754388]


In [100]:
equity_portfolio.equity_share[asset_id_tmp].spread_country = 0.01
equity_portfolio.equity_share[asset_id_tmp].spread_sector = 0
equity_portfolio.equity_share[asset_id_tmp].spread_stress = 0.01

In [101]:
price2 = equity_portfolio.equity_share[asset_id_tmp].price_share(temp_dividend, temp_terminal, settings.modelling_date, proj_period, curves)
print(price2)

[89.59754388]


In [102]:
equity_portfolio.equity_share[asset_id_tmp].spread_country = 0
equity_portfolio.equity_share[asset_id_tmp].spread_sector = 0
equity_portfolio.equity_share[asset_id_tmp].spread_stress = 0.02


In [103]:
price3 = equity_portfolio.equity_share[asset_id_tmp].price_share(temp_dividend, temp_terminal, settings.modelling_date, proj_period, curves)
print(price3)

[89.59754388]


## Calibration of an equity share

In a real world run, the growth rate of each equity is brough in as a modelling assumption and must be calculated externaly. If the user is interested in a risk-neutral run, the performance of each asset must equal to that of the risk free rate. In OSEM this is done using the growth rate as the calibrating parameter. To do this, a bisection algorithm is used to calibrate the growth rate such that the discounted cash flows of each equity equal to the current market price.

For this demonstration, a single equity position is created, the growth rate is calibrated and then the present value of the calibrated equity is compared to the assumed current market value.

Note that the projection period is selected as 0 meaning the initial modelling date. But the calibration works also for other projection periods. The only change needed would be to change the market price.



In [104]:
proj_period = 0

In [105]:
spread_country = 0
spread_sector = 0.02
spread_stress = 0

In [106]:
test_share_1 = EquityShare(asset_id=1, nace='A1.4.5', issuer=None, issue_date=datetime.date(2021, 12, 3), dividend_yield=0.03, frequency=1, units=1,market_price=94.0, growth_rate=0.01,spread_country=spread_country,spread_sector=spread_sector,spread_stress=spread_stress )

In [107]:
opt_growth = test_share_1.bisection_growth(-1, 1, settings.modelling_date, settings.end_date, proj_period, curves, 0.00000001,100000)

In [108]:
print(opt_growth)

0.023426063358783722


In [109]:
test_dividends = test_share_1.create_single_cash_flows(settings.modelling_date, settings.end_date, opt_growth)
test_terminal = test_share_1.create_single_terminal(settings.modelling_date, settings.end_date, curves.ufr, opt_growth)

If the calibration was performed correctly, the present value calculated by discounting future cash flows with the assumed risk free rate should be equal to the initial observed market price.

In [110]:
print(test_share_1.market_price)

94.0


In [111]:
print(test_share_1.price_share(test_dividends, test_terminal, settings.modelling_date, proj_period, curves))

[93.99999919]
